# Experiment 20: Fine-Tune GPT-2 for a Specific Text Style and Generate Text

**Objective:** Load a pre-trained Large Language Model (GPT-2), fine-tune it on a custom text style dataset, and generate text based on user input.

**Language:** Python  
**Estimated Time:** 2 hours  
**Libraries Used:** transformers, datasets, torch, pandas


## 1. Introduction

GPT-2 is a pre-trained transformer-based language model. It can generate natural language text, and with fine-tuning, it can learn a custom writing style such as poetry, formal writing, technical writing, stories, quotes, or social media text.

In this experiment, we will:

1. Load a pre-trained GPT-2 model
2. Prepare a custom dataset containing a particular writing style
3. Tokenize the text
4. Fine-tune GPT-2 on that dataset
5. Save the model
6. Generate new text from user prompts


## 2. Install Required Libraries


In [ ]:
# Run this once if packages are not installed
# !pip install transformers datasets accelerate torch pandas

## 3. Import Libraries


In [ ]:
import os
import pandas as pd
import torch

from datasets import Dataset
from transformers import (
    GPT2Tokenizer,
    GPT2LMHeadModel,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

torch.manual_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)

## 4. Create or Load a Style Dataset

You can replace the sample texts below with your own dataset. For better fine-tuning, provide many examples in the style you want the model to learn.


In [ ]:
# Example custom-style dataset
style_texts = [
    'The moon slipped quietly behind the clouds, and the village slept under a silver silence.',
    'In the old forest, every tree remembered a story the wind had once whispered.',
    'She wrote letters to the rain, hoping the sky would answer in thunder.',
    'The river carried secrets that only the night was patient enough to hear.',
    'A lantern glowed at the window like a small promise against the dark.',
    'Dreams arrived softly, wrapped in the scent of wet earth and jasmine.',
    'He walked through the dawn as if the light itself had been waiting for him.',
    'Even silence became music when the stars leaned close to listen.'
]

df = pd.DataFrame({'text': style_texts})
df.head()

## 5. Convert DataFrame to Hugging Face Dataset


In [ ]:
dataset = Dataset.from_pandas(df)
dataset

## 6. Load GPT-2 Tokenizer and Model

We use `gpt2` as the base model. GPT-2 does not have a pad token by default, so we set the pad token equal to the end-of-sequence token.


In [ ]:
model_name = 'gpt2'

tokenizer = GPT2Tokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = GPT2LMHeadModel.from_pretrained(model_name)
model.resize_token_embeddings(len(tokenizer))
model.config.pad_token_id = tokenizer.pad_token_id
model.to(device)

print('Tokenizer and model loaded successfully.')

## 7. Tokenize the Dataset


In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        truncation=True,
        padding='max_length',
        max_length=64,
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=['text'])
tokenized_dataset

## 8. Data Collator

For causal language modeling, labels are created from the input tokens themselves.


In [ ]:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

## 9. Training Arguments

These settings keep the training small and simple for lab use. Increase epochs and dataset size for better results.


In [ ]:
training_args = TrainingArguments(
    output_dir='./gpt2_finetuned_style',
    overwrite_output_dir=True,
    num_train_epochs=5,
    per_device_train_batch_size=2,
    save_steps=50,
    save_total_limit=1,
    logging_steps=10,
    learning_rate=5e-5,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    report_to='none'
)

## 10. Fine-Tune GPT-2


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

trainer.train()

## 11. Save the Fine-Tuned Model


In [ ]:
save_path = './gpt2_finetuned_style_final'
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)
print('Fine-tuned model saved at:', save_path)

## 12. Generate Text from User Input

This cell takes a prompt from the user and generates text in the learned style.


In [ ]:
def generate_text(prompt, max_length=80):
    inputs = tokenizer(prompt, return_tensors='pt', padding=True, truncation=True).to(device)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            max_length=max_length,
            num_return_sequences=1,
            no_repeat_ngram_size=2,
            do_sample=True,
            top_k=50,
            top_p=0.95,
            temperature=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )

    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

# Example prompt
prompt = 'The evening sky'
generated_text = generate_text(prompt)
print('Prompt:', prompt)
print('\nGenerated Text:\n')
print(generated_text)

## 13. Interactive User Input

Run this cell and type your own prompt.


In [ ]:
user_prompt = input('Enter your prompt: ')
print('\nGenerated Output:\n')
print(generate_text(user_prompt))

## 14. Notes for Better Fine-Tuning

- Use a much larger dataset for better style learning.
- Clean and format the text consistently.
- Increase training epochs if loss is still high.
- You can replace the sample texts with poetry, technical paragraphs, dialogues, news, or any other style.
- Fine-tuning large models may require a GPU for faster training.


## 15. Conclusion

In this experiment, we loaded the pre-trained GPT-2 model, fine-tuned it on a small custom-style dataset, and generated text from user prompts. This demonstrates how a general language model can be adapted to produce content in a more specific writing style.


## 16. Viva Questions

1. What is GPT-2?
2. What is fine-tuning in language models?
3. Why do we set the pad token in GPT-2?
4. What is the purpose of tokenization?
5. How can text generation quality be improved?
